In [67]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


In [68]:
course_catalog_data = pd.read_csv('uvm_courses_standardized_11_20_25.csv')
course_catalog_data.head()

/var/folders/yz/s7yhnqx575nf_63f35422l8m0000gn/T/ipykernel_55544/1355856728.py:1: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  course_catalog_data = pd.read_csv('uvm_courses_standardized_11_20_25.csv')


,year,cat_type,ipeds_id,department,catalog_prefix,course_number,title,instructors,cross_listed_with,is_subcourse,...,is_continued_from_previous,full_entry,first_instructor,num_instructors,prefix_cleaned,department_cleaned,prefix_cleaned_from_dept,prefix_cleaned_from_prefix,department_cleaned_from_prefix,department_cleaned_from_dept
0,1965,gr,231174,economics,ECON,276,C.P.A. PROBLEMS,['Mr. Nyquist'],[],False,...,False,276 C.P.A. PROBLEMS Review of questions and pr...,Mr. Nyquist,1,EC,economics,EC,EC,economics,economics
1,1965,gr,231174,economics,ECON,286,ECONOMIC ANALYSIS,['Mr. Wass'],[],False,...,False,"286 ECONOMIC ANALYSIS Analysis of demand, supp...",Mr. Wass,1,EC,economics,EC,EC,economics,economics
2,1965,gr,231174,economics,ECON,288,QUALITY CONTROL,['Mr. Saunders'],[],False,...,False,288 QUALITY CONTROL The application of statist...,Mr. Saunders,1,EC,economics,EC,EC,economics,economics
3,1965,gr,231174,economics,ECON,290,THE SOVIET ECONOMY,['Mr. Dellin'],[],False,...,False,290 THE SOVIET ECONOMY An analysis of the econ...,Mr. Dellin,1,EC,economics,EC,EC,economics,economics
4,1965,gr,231174,economics,ECON,291,ECONOMIC PATTERNS AND POLICIES OF EASTERN EUROPE,['Mr. Dellin'],[],False,...,False,291 ECONOMIC PATTERNS AND POLICIES OF EASTERN ...,Mr. Dellin,1,EC,economics,EC,EC,economics,economics


In [69]:
print(course_catalog_data.columns)
print(course_catalog_data.shape)
print(course_catalog_data["department_cleaned"].unique())

Index(['year', 'cat_type', 'ipeds_id', 'department', 'catalog_prefix',
       'course_number', 'title', 'instructors', 'cross_listed_with',
       'is_subcourse', 'is_truncated', 'is_continued_from_previous',
       'full_entry', 'first_instructor', 'num_instructors', 'prefix_cleaned',
       'department_cleaned', 'prefix_cleaned_from_dept',
       'prefix_cleaned_from_prefix', 'department_cleaned_from_prefix',
       'department_cleaned_from_dept'],
      dtype='object')
(222768, 21)
['economics' 'civil & environmental engineering' 'latin' 'mathematics'
 'chemistry' 'chinese' 'home economics' 'history' 'political science'
 'communication and theatre' 'military studies' 'education'
 'physical education' 'zoology' 'psychology' 'sociology' 'art'
 'natural resources' 'elementary education' 'engineering'
 'agricultural and resource economics' 'hebrew' 'general literature'
 'botany' 'greek' 'greek & latin' 'spanish' 'forestry' 'geography'
 'business administration' 'anthropology' 'english' 

In [70]:
import re
import pandas as pd

# --- Boilerplate removal patterns ---
BOILERPLATE_PATTERNS = [
    # prereq / coreq / permissions
    r"\bPre/?co-?requisites?\b\s*:\s*.*?(?=\.|$)",
    r"\bPre-?requisites?\b\s*:\s*.*?(?=\.|$)",
    r"\bPrerequisites?\b\s*:\s*.*?(?=\.|$)",
    r"\bCorequisites?\b\s*:\s*.*?(?=\.|$)",
    r"\bCo-?requisites?\b\s*:\s*.*?(?=\.|$)",
    r"\bRecommended\b\s*:\s*.*?(?=\.|$)",
    r"\bDepartmental\s+permission\b.*?(?=\.|$)",
    r"\bInstructor\s+Permission\b.*?(?=\.|$)",

    # credits / hours / contact formats
    r"\bCredits?\b\s*:\s*\d+(?:\s*-\s*\d+)?\b\.?",
    r"\b\d+(?:\s*-\s*\d+)?\s*Credits?\b\.?",          # "3 Credits." or "1-18 Credits"
    r"\bcredit\s+arranged\b.*?(?=\.|$)",
    r"\b(One|Two|Three|Four|Five|Six|Seven|Eight|Nine|Ten)\s+hours?\b\.?",

    # catalog notes
    r"\bAlternate\s+years?\b.*?(?=\.|$)",
    r"\bSee\s+Schedule\s+of\s+Courses\b.*?(?=\.|$)",
    r"\bIn\s+collaboration\b.*?(?=\.|$)",
]

# Instructor at end, with or without preceding period (covers "Mr. Nyquist.", "Staff.", "Zimmerman.", "Dr. Johnstone.")
INSTRUCTOR_AT_END = re.compile(
    r"(?:[\.\s]+)(?:Staff|(?:Mr|Ms|Mrs|Dr)\.?\s+[A-Z][a-z]+|[A-Z][a-z]+)\.?\s*$"
)

# Header: optional subject code, course number(s), then title, then rest
HEADER_RE = re.compile(
    r"""^\s*
        (?:(?P<subject>[A-Z&]{2,6})\s*)?              # e.g., HDFS, GRMD, ORTH, CHIN (optional)
        (?P<number>\d{1,3}(?:\s*,\s*\d{1,3})*)        # e.g., 297, 298 or 276
        [\.\s]+
        (?P<title>.+?)\s+
        (?P<rest>.+)$
    """,
    re.VERBOSE | re.DOTALL,
)

def normalize_text(x) -> str:
    """Normalize whitespace and repair hyphenation artifacts."""
    if x is None or pd.isna(x):
        return ""

    s = str(x)

    # Fix hyphenation that came from line wraps: "func-\n tion" -> "function"
    s = re.sub(r"-\s*\n\s*", "", s)

    # Replace remaining newlines with spaces
    s = re.sub(r"\s*\n\s*", " ", s)

    # Fix in-line broken hyphenation like "immuni- ty" or "auto- mobile"
    # Only join when the right side starts with lowercase (so we don't mangle "T-cell", "Major Histocompatibility")
    s = re.sub(r"\b([A-Za-z]{2,})-\s+([a-z]{2,})\b", r"\1\2", s)

    # Collapse whitespace
    s = re.sub(r"\s+", " ", s).strip()
    return s

def extract_main_text(raw) -> str:
    """
    Return the main descriptive text only (drops title/number, prerequisites, credits, hours, instructors).
    """
    s = normalize_text(raw)
    if not s:
        return ""

    # Remove obvious trailing instructor tokens
    s = INSTRUCTOR_AT_END.sub("", s)

    # Pull body after the title (this avoids the partial-title junk you were seeing)
    m = HEADER_RE.match(s)
    body = m.group("rest").strip() if m else s

    # Remove parenthetical contact pattern "(3-0)" or "(2-2)" or similar
    body = re.sub(r"\(\s*\d+\s*-\s*\d+\s*\)", " ", body)

    # Remove boilerplate segments
    for pat in BOILERPLATE_PATTERNS:
        body = re.sub(pat, " ", body, flags=re.IGNORECASE)

    # Remove standalone trailing credits/hours if they slipped through (common edge case)
    body = re.sub(r"\b\d+(?:\s*-\s*\d+)?\s*Credits?\b\.?\s*$", "", body, flags=re.IGNORECASE)
    body = re.sub(r"\b(One|Two|Three|Four|Five|Six|Seven|Eight|Nine|Ten)\s+hours?\b\.?\s*$", "", body, flags=re.IGNORECASE)

    # Final cleanup
    body = re.sub(r"\s+", " ", body).strip()
    body = re.sub(r"\s+\.", ".", body)
    body = body.strip(" .;:-")

    return body


In [71]:
course_catalog_data["main_text"] = course_catalog_data["full_entry"].apply(extract_main_text)

In [72]:
print(course_catalog_data["full_entry"].iloc[0])
print(course_catalog_data["full_entry"].iloc[10])
print(course_catalog_data["full_entry"].iloc[10000])
print(course_catalog_data["full_entry"].iloc[20000])
print(course_catalog_data["full_entry"].iloc[30000])
print(course_catalog_data["full_entry"].iloc[40000])
print(course_catalog_data["full_entry"].iloc[50000])
print(course_catalog_data["full_entry"].iloc[60000])
print(course_catalog_data["full_entry"].iloc[70000])
print(course_catalog_data["full_entry"].iloc[120000])
print(course_catalog_data["full_entry"].iloc[130000])
print(course_catalog_data["full_entry"].iloc[140000])
print(course_catalog_data["full_entry"].iloc[150000])
print(course_catalog_data["full_entry"].iloc[200000])
print(course_catalog_data["full_entry"].iloc[210000])
print(course_catalog_data["full_entry"].iloc[220000])
print(course_catalog_data["full_entry"].iloc[222600])

276 C.P.A. PROBLEMS Review of questions and problems from past C.P.A. examinations, including partnerships, corporations, financial statements, auditing, cost accounting, insolvencies, receiverships, liquidations, consolidations, estates, trusts, governmental and institutional accounting methods. Prerequisite: 162. Three hours. Mr. Nyquist.
297, 298 SEMINAR Review of recent books and periodical literature; discussions and reports on topics of contemporary interest. Prerequisite: permission of the department. Three hours. Staff.
10 Automobile Basics (3-0) Basic course in auto-mobile mechanics, management, ownership, and opera-tion. Society related issues such as energy, pollution, and safety also discussed. Three hours. Zimmerman.
225 Number Theory for Teachers Division algorithm, prime numbers, fundamental theorem of arithmetic, factors and multiples, number bases, arithmetic progressions; emphasis on how number theory is taught in grades K-8. Pre/co-requisites: MAED 205, 210, and 215.

In [73]:
print(course_catalog_data["main_text"].iloc[0])
print(course_catalog_data["main_text"].iloc[10])
print(course_catalog_data["main_text"].iloc[10000])
print(course_catalog_data["main_text"].iloc[20000])
print(course_catalog_data["main_text"].iloc[30000])
print(course_catalog_data["main_text"].iloc[40000])
print(course_catalog_data["main_text"].iloc[50000])
print(course_catalog_data["main_text"].iloc[60000])
print(course_catalog_data["main_text"].iloc[70000])
print(course_catalog_data["main_text"].iloc[120000])
print(course_catalog_data["main_text"].iloc[130000])
print(course_catalog_data["main_text"].iloc[140000])
print(course_catalog_data["main_text"].iloc[150000])
print(course_catalog_data["main_text"].iloc[200000])
print(course_catalog_data["main_text"].iloc[210000])
print(course_catalog_data["main_text"].iloc[220000])
print(course_catalog_data["main_text"].iloc[222600])

PROBLEMS Review of questions and problems from past C.P.A. examinations, including partnerships, corporations, financial statements, auditing, cost accounting, insolvencies, receiverships, liquidations, consolidations, estates, trusts, governmental and institutional accounting methods
Review of recent books and periodical literature; discussions and reports on topics of contemporary interest
Basics Basic course in auto-mobile mechanics, management, ownership, and opera-tion. Society related issues such as energy, pollution, and safety also discussed
Theory for Teachers Division algorithm, prime numbers, fundamental theorem of arithmetic, factors and multiples, number bases, arithmetic progressions; emphasis on how number theory is taught in grades K-8
GEOGRAPHY OF EUROPE (Geography 202 same as History 202) European geography within a framework of past times, the historical development and distribution of settlement, economic and political patterns
THEORY Lattices and Boolean algebras, 

In [74]:
course_catalog_data["is_graduate"] = course_catalog_data["cat_type"].eq("gr")

# You already have: course_catalog_data (222,768 x 21)
df = course_catalog_data.copy()

# Pick the text field you want to analyze:
TEXT_COL = "main_text"
DEPT_COL = "department_cleaned"  # or "department"

# grad flag (adjust this if your cat_type coding differs)
df["is_grad"] = df["cat_type"].astype(str).str.lower().eq("gr")

# global corpora (across all departments)
grad_text = " ".join(df.loc[df["is_grad"], TEXT_COL].dropna().astype(str).tolist())
ug_text   = " ".join(df.loc[~df["is_grad"], TEXT_COL].dropna().astype(str).tolist())

# OPTIONAL: corpora per department (a dict dept -> text)
grad_by_dept = df[df["is_grad"]].groupby(DEPT_COL)[TEXT_COL].apply(
    lambda s: " ".join(s.dropna().astype(str))
).to_dict()

ug_by_dept = df[~df["is_grad"]].groupby(DEPT_COL)[TEXT_COL].apply(
    lambda s: " ".join(s.dropna().astype(str))
).to_dict()


In [75]:
import re
from collections import Counter
import numpy as np

WORD_RE = re.compile(r"[a-z]+(?:'[a-z]+)?")  # simple word tokens

def text_to_ranked_df(text: str, *, min_count: int = 1) -> pd.DataFrame:
    tokens = WORD_RE.findall(text.lower())
    c = Counter(tokens)

    items = [(w, n) for w, n in c.items() if n >= min_count]
    items.sort(key=lambda x: x[1], reverse=True)

    out = pd.DataFrame(items, columns=["types", "counts"])
    total = out["counts"].sum()
    out["total_unique"] = len(out)
    out["probs"] = out["counts"] / total if total > 0 else 0.0
    return out


In [76]:
grad_df = text_to_ranked_df(grad_text, min_count=2)
ug_df   = text_to_ranked_df(ug_text,   min_count=2)


In [77]:
import json

def write_allotax_json(df_ranked: pd.DataFrame, out_path: str) -> None:
    keep = df_ranked[["types", "counts", "total_unique", "probs"]].copy()

    # rename to match py_allotax requirement
    keep = keep.rename(columns={"total_unique": "totalunique"})

    # enforce plain json-friendly types
    keep["types"] = keep["types"].astype(str)
    keep["counts"] = keep["counts"].astype(int)
    keep["totalunique"] = keep["totalunique"].astype(int)
    keep["probs"] = keep["probs"].astype(float)

    records = keep.to_dict(orient="records")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False)


In [78]:
write_allotax_json(grad_df, "grad.json")
write_allotax_json(ug_df,   "ug.json")


In [81]:
from py_allotax.generate_svg import generate_svg

generate_svg(
    "ug.json",
    "grad.json",
    "ug_vs_grad.html",   # despite the name, they show .pdf in the README
    "0.17",             # alpha
    "Undergrad",
    "Graduate",
    desired_format="html"
)


HTML saved to ug_vs_grad.html


In [83]:
# Create allotax plots for selected departments comparing 1995-2000 vs 2015-2020
departments_to_plot = ['community development and applied economics',
                       'mathematics', 'engineering', 'biology', 'chemistry', 'computer science']

df = course_catalog_data.copy()
# Filter data for 1995-2000 and 2015-2020
df_early = df[(df['year'] >= 1995) & (df['year'] <= 2000)].copy()
df_late = df[(df['year'] >= 2015) & (df['year'] <= 2020)].copy()

print("Creating allotax plots for department evolution (1995-2000 vs 2015-2020)")
print("="*70)

for dept in departments_to_plot:
    # Get courses for each period
    courses_early = df_early[df_early[DEPT_COL] == dept]
    courses_late = df_late[df_late[DEPT_COL] == dept]

    # Get text for each period
    text_early = " ".join(
        courses_early[TEXT_COL].dropna().astype(str).tolist()
    )
    text_late = " ".join(
        courses_late[TEXT_COL].dropna().astype(str).tolist()
    )

    # Skip if no data for either period
    if not text_early or not text_late:
        print(f"Skipping {dept}: insufficient data for 1995-2000 or 2015-2020")
        continue

    # Show course counts
    print(f"\n{dept}:")
    print(f"  1995-2000: {len(courses_early)} courses")
    print(f"  2015-2020: {len(courses_late)} courses")

    # Text length diagnostics for early period
    text_early_series = courses_early[TEXT_COL].dropna().astype(str)
    if len(text_early_series) > 0:
        print(f"\n  1995-2000 Text Length Statistics:")
        print(f"    {text_early_series.str.len().describe().to_string().replace(chr(10), chr(10) + '    ')}")
        print(f"    Proportion < 60 chars: {(text_early_series.str.len() < 60).mean():.2%}")

    # Text length diagnostics for late period
    text_late_series = courses_late[TEXT_COL].dropna().astype(str)
    if len(text_late_series) > 0:
        print(f"\n  2015-2020 Text Length Statistics:")
        print(f"    {text_late_series.str.len().describe().to_string().replace(chr(10), chr(10) + '    ')}")
        print(f"    Proportion < 60 chars: {(text_late_series.str.len() < 60).mean():.2%}")

    # Sample entries from early period
    if len(text_early_series) >= 3:
        samples = text_early_series.sample(min(3, len(text_early_series)), random_state=0).tolist()
        print(f"\n  1995-2000 Sample entries:")
        for i, sample in enumerate(samples, 1):
            print(f"    {i}. {sample[:150]}{'...' if len(sample) > 150 else ''}")

    # Create ranked dataframes
    df_early_ranked = text_to_ranked_df(text_early, min_count=1)
    df_late_ranked = text_to_ranked_df(text_late, min_count=1)

    print(f"  1995-2000: {len(df_early_ranked)} unique words")
    print(f"  2015-2020: {len(df_late_ranked)} unique words")

    # Write JSON files
    json_early = f"{dept.replace(' ', '_')}_1995.json"
    json_late = f"{dept.replace(' ', '_')}_2020.json"
    write_allotax_json(df_early_ranked, json_early)
    write_allotax_json(df_late_ranked, json_late)

    # Generate allotax plot
    output_file = f"{dept.replace(' ', '_')}_1995_vs_2020.html"

    try:
        generate_svg(
            json_early,
            json_late,
            output_file,
            "0.17",  # alpha parameter
            f"{dept} 1995-2000",
            f"{dept} 2015-2020",
            desired_format="html"
        )
        print(f"  ✓ Created {output_file}")
    except Exception as e:
        print(f"  ✗ Failed to create {output_file}: {e}")

print("\n" + "="*70)
print("Allotax plots generation complete!")
print("Files saved in current directory with pattern: <DEPT>_1995_vs_2020.html")

Creating allotax plots for department evolution (1995-2000 vs 2015-2020)

community development and applied economics:
  1995-2000: 373 courses
  2015-2020: 496 courses

  1995-2000 Text Length Statistics:
    count    373.000000
    mean     177.238606
    std       65.666225
    min       15.000000
    25%      147.000000
    50%      182.000000
    75%      222.000000
    max      361.000000
    Proportion < 60 chars: 8.31%

  2015-2020 Text Length Statistics:
    count    496.000000
    mean     182.056452
    std       77.267114
    min        6.000000
    25%      153.000000
    50%      191.000000
    75%      222.000000
    max      368.000000
    Proportion < 60 chars: 12.30%

  1995-2000 Sample entries:
    1. Industrial Production Principles, concepts, methods employed in organizing capital, labor, tools, machines for producing products. Students function a...
    2. in Economic Development Role of agriculture in development of less-developed countries. Discussion of alterna